# Full analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/clarkmit/abcd-biopsychosocial-prediction/blob/main/notebooks/02_full_analysis.ipynb)

The complete pipeline, one section per stage, in the same order as the published notebook (`paper/ABCDMLPipeline_v1.0.0.ipynb`). Settings that were Colab form fields are configuration fields here (`cfg.<stage>.<setting>`).

ABCD data require an approved Data Use Certification. Point `DATA_PATH` at your merged panel outside this repository, and never upload ABCD data to shared services. By default this notebook uses the fully simulated panel shipped in `data/simulated/`, whose columns are independent, so the metrics stay near zero and the run only checks that every stage works. With the reduced settings below, all stages take about 45 minutes on a laptop CPU. Publication-scale settings take far longer (see `docs/reproducing.md`).

In [ ]:
# Find the repository (or fetch it when running on Colab) and install the package.
import importlib, subprocess, sys
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "demo.yaml").exists()), None)
if ROOT is None:  # e.g. Google Colab
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/clarkmit/abcd-biopsychosocial-prediction"], check=True)
    ROOT = Path.cwd() / "abcd-biopsychosocial-prediction"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(ROOT)], check=True)
    importlib.invalidate_caches()  # make the newly installed package importable in this session
print("repository:", ROOT)

## Configuration

In [ ]:
from bpsp import Pipeline, PipelineConfig, ctx

cfg = PipelineConfig()                                   # published defaults
cfg.setup.DATA_PATH = str(ROOT / "data/simulated/CLEAN_ABCD_5_1_panel_simulated.csv")
cfg.setup.objective_subjective_spectrum = False
cfg.output_dir = str(ROOT / "outputs" / "full_analysis")

cfg.main.target_options = "internal_D_p"
cfg.main.split = "across_categories"
cfg.main.tp_option = "2"
cfg.main.batch_mode = "Single Target"
cfg.main.skip_tabpfn = True

# Smaller models and counts for a quick pass; remove for publication-scale runs.
cfg.model_overrides = {"catboost": {"iterations": 100}, "xgboost": {"n_estimators": 100},
                       "random_forest": {"n_estimators": 100}}
cfg.validate()
pipe = Pipeline(cfg)
pipe.setup()

## Main analysis

Stacking ensemble for the target, weighted consensus feature importance and SHAP.

In [ ]:
pipe.run_stage("main")

## Batch summary

Summary table across targets (classification batch runs).

In [ ]:
pipe.run_stage("batch_summary")

## Sankey diagrams

Across-category feature-to-target architecture (batch runs).

In [ ]:
pipe.run_stage("sankey")

## Feature networks

EBICglasso partial correlation network, correlation matrix and dendrogram of the top 10 consensus features.

In [ ]:
cfg.networks.n_bootstrap = 200
pipe.run_stage("networks")

## Model diagnostics

Model diversity, bootstrap stability, learning curves, SHAP beeswarm and effect sizes.

In [ ]:
cfg.diagnostics.n_bootstraps = 20
pipe.run_stage("diagnostics")

## Smoke test

A quick end-to-end check that the stacking model can be created, fitted and used to predict on 200 training rows.

In [ ]:
pipe.run_stage("smoke_test")

## Nested cross-validation (S1.4, S1.11.3, S1.11.10)

Nested cross-validation with 95% confidence intervals, calibration and decision curves.

In [ ]:
cfg.nested_cv.n_outer_folds = 5
pipe.run_stage("nested_cv")

## Demographic stratification (S1.17, S1.11.7)

Extended demographic-stratified performance (child sex, parent sex, parent education, SES tertile, adverse life event exposure).

In [ ]:
cfg.stratified.n_bootstrap = 200
pipe.run_stage("stratified")

## Family- and site-grouped cross-validation (S1.5, S1.18.4)

Whole families or sites held out in each fold.

In [ ]:
cfg.grouped_cv.robustness_mode = "Both"
pipe.run_stage("grouped_cv")

## Permutation testing (S1.4.4, S1.11.5)

Permutation testing with Phipson-Smyth correction, using a single CatBoost model.

In [ ]:
cfg.permutation.n_permutations = 50
pipe.run_stage("permutation")

## Stacking audit (S1.1)

Leakage checks: every row in exactly one inner validation fold, meta-learner fit only on out-of-fold predictions.

In [ ]:
pipe.run_stage("stacking_audit")

## PCA baseline and temporal hold-out (S1.3, S1.6)

PCA baseline comparisons and temporal hold-out generalization.

In [ ]:
pipe.run_stage("baselines")

## Domain ablation (S1.19)

Domain ablation and feature-domain importance.

In [ ]:
pipe.run_stage("ablation")

## Cross-timepoint trajectories (S1.16.1)

Concurrent performance at each wave.

In [ ]:
pipe.run_stage("trajectories")

## Temporal gradient (S1.16.2)

Concurrent versus prospective decay.

In [ ]:
pipe.run_stage("temporal_gradient")

## Race/ethnicity and religion (S5.1)

Performance by race/ethnicity and religion, with partial correlation adjustment.

In [ ]:
cfg.fairness.frr_n_bootstrap = 200
pipe.run_stage("fairness")

## Calibration and recalibration (S5.2)

Calibration and Platt and isotonic recalibration.

In [ ]:
cfg.calibration.rcal_n_bootstrap = 200
pipe.run_stage("calibration")

## Base learners vs ensemble (S5.3)

Each base learner against the stacked ensemble on held-out data.

In [ ]:
pipe.run_stage("base_learners")

## Grouped permutation importance (S5.4)

Grouped permutation importance for correlated and change-based features.

In [ ]:
cfg.grouped_importance.gpi_n_repeats = 3
pipe.run_stage("grouped_importance")

## Timings

Seconds per stage.

In [ ]:
pipe.timings

## Interpreting the output

On the simulated panel the models have nothing to learn, so metrics near zero are the expected result. For a run with a known answer, see `01_quickstart.ipynb`.

With ABCD data, the settings for each figure and supplementary analysis of the article are listed in `docs/reproducing.md`, and `configs/paper_defaults.yaml` holds the published defaults. Results describe population-level associations. They are not causal and not for decisions about individuals (`docs/model_card.md`).